# ShrAdd2 — 2-Party Additive Secret Sharing (EMP2)

ELL ∈ [2, 31].  Point-to-point between two parties over a single
Channel.  Used in MPMT for the Querier ↔ Leader hash phase and
equality-test phase.

## Factory

``ShrAdd2(ell, party)`` returns a type.  Call the factory with a
``Channel`` to construct a protocol instance.
Party 0 and Party 1 exchange data over the same TCP connection.


In [ ]:
# This cell requires multi-party setup and will not run in a notebook

import mpmt
from mpmt.channels import Channel

Add2_0 = mpmt.ShrAdd2(ell=14, party=0)
Add2_1 = mpmt.ShrAdd2(ell=14, party=1)

# Party 0 (server):
inst0 = Add2_0(Channel(port=14000))

# Party 1 (client):
inst1 = Add2_1(Channel(host="127.0.0.1", port=14000))


## Ring-Addition Share

``inst.share_scalar(val)`` — P0 shares a scalar; P1 calls
``inst.recv_scalar_share()``.  Both return the local ADD2 share
component (a ``uint32_t``).  The two shares sum to *val* mod 2^ELL.

Also: ``inst.send_data(val)`` / ``inst.recv_data()`` for plain
transfers.


In [ ]:
# This cell requires multi-party setup and will not run in a notebook

# P0:
my_share = inst0.share_scalar(val=42)

# P1:
peer_share = inst1.recv_scalar_share()

# Reconstructed: (my_share + peer_share) mod 2^ell == 42


## XOR Share — Element (variable-length)

``inst.share_element(plain)`` — XOR-shares a variable-length element.
Sends ``[uint32_t len][share bytes]`` over the wire.  Returns the
local XOR share as ``bytes``.

``inst.recv_element_share()`` — receives the peer's XOR share.
Returns ``bytes``.

The two XOR shares XOR together to recover the plaintext.  Used in
the query hash: Querier shares the element, Leader receives.


In [ ]:
# This cell requires multi-party setup and will not run in a notebook

# Querier (party=1):
e_share = inst1.share_element(b"alice")

# Leader (party=0):
e_share = inst0.recv_element_share()


## XOR Share — Key (16-byte)

``inst.share_key(key)`` — XOR-shares a 16-byte AES key.
Sends only the share bytes (no length prefix).  ``key`` must be exactly
16 bytes.

``inst.recv_key_share(buf)`` — receives into a pre-allocated
``bytearray(16)``.

Used for hash seeds: Leader shares each seed, Querier receives.


In [ ]:
# This cell requires multi-party setup and will not run in a notebook

# Leader (party=0):
inst0.share_key(seed_16bytes)

# Querier (party=1):
buf = bytearray(16)
inst1.recv_key_share(buf)


## GC Operations

All three run inside a garbled circuit between the two parties.
Both must call them in the same order.

### hash

``inst.hash(my_pt, my_key)`` — AES-DM hash.  *my_pt* is the local
XOR share of the preimage (``bytes``, ≤ 16 bytes).  *my_key* is the
local XOR share of the key (exactly 16 bytes).  Returns a ``uint32_t``
ADD2 share of the hash result.

### mod

``inst.mod(my_a, mv)`` — ``(a_0 + a_1) % mv`` computed in-circuit.
*my_a* is the local ADD2 share; *mv* is the public modulus.  Returns
a ``uint32_t`` ADD2 share of the result.

### equality_test

``inst.equality_test(my_a, my_b)`` — compares two ADD2-shared values
in-circuit.  Returns a ``uint32_t`` ADD2 share: sum of both parties'
shares = 1 if equal, 0 otherwise.

In the Emp2 implementation, the result is masked and revealed only to
Bob (party=1), so only the Querier can reconstruct the final answer.


In [ ]:
# This cell requires multi-party setup and will not run in a notebook

# Both parties in sync:
hr = inst.hash(my_pt_share, my_key_share)
mr = inst.mod(hr, bf_size)
et = inst.equality_test(my_dot_share, 0)


## Query Protocol Usage

**Hash phase** (Leader P0 ↔ Querier P1, ell=ell_add2):
1. Querier share_element → Leader recv_element_share
2. For each of hf_num seeds: Leader share_key → Querier recv_key_share
3. Both: hash(preimage_share, key_share) → mod(result, bf_size)
4. Querier sends ADD2 index shares to Helpers (RingTransport)

**ET phase** (Leader P0 ↔ Querier P1, ell=ell_query):
1. Leader: equality_test(dot_share_L, hf_num)
2. Querier: equality_test(dot_share_Q, 0)
3. Leader sends its ET share to Querier (RingTransport)
4. Querier: ring_add → 0 or 1


## Byte Counters

- ``inst.bytes_sent()``, ``inst.bytes_recv()``
- ``inst.clear_send_cnt()``, ``inst.clear_recv_cnt()``
